<a href="https://colab.research.google.com/github/francescopatane96/eNERVE/blob/main/1_proteins_data_mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1: Data mining 

This bioinformatic project lets us to retrieve all data from uniprot about every reviewed protein deposited in Uniprot database (Uniprot release 2022_04).

From this release we will obtain a dataset containing the subcellular localization of every reviewed protein.

Our scope is to obtain from every of these proteins, features like the 'Organism', 'Sequence' and ' Subcellular localization'.

Then, we will want to compare organisms from Uniprot between a list of organisms of interest (obtained from VEUpathDB Database). This is perform because we want to create a machine learning predictor about the subcellular localization of eukaryotic pathogenic proteins in order to make this system part of eNERVE, a reverse vaccinology predictor software. 

Finally, we will drop every row from dataframe created using uniprot proteins, that doesn't appear in VEUpathDB list of organisms.

1. Note that the csv file was modified using Calc. In fact, in ca 100.000 rows there were some commas. It is better to remove them, using calc (libreoffice) command 'find and replace', with for example, a dot(.).

In [ ]:
import pandas as pd     #import dependencies 
from tqdm import tqdm    #progress bar
import csv

In [ ]:
#https://veupathdb.org/veupathdb/app/step/429610153/download         #eukaryotic pathogenic organisms (it includes hosts and vectors)

In [ ]:
#all reviewed proteins from uniprot
#https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.xml.gz

In [ ]:
#upload the uniprot csv and check it for bad lines

allproteins_df = pd.read_csv('allproteins_table.csv', header=0, sep=',',
                             engine='python', encoding='utf-8', error_bad_lines=False,
                             quoting=csv.QUOTE_NONE)


In [ ]:
allproteins_df.count()

In [ ]:
#dropNaN
allproteins_df = allproteins_df[['Entry Name', 'Organism', 'Subcellular location', 'Sequence']].dropna()

In [ ]:
df_reindex_allproteins = allproteins_df.reset_index()
proteins_df = df_reindex_allproteins.drop(columns='index')
print(proteins_df)

In [ ]:
#retains only first two words in 'Organism' column, mandatory to compare the column with organisms of interests list
proteins_df['Organism'] = proteins_df['Organism'].apply(lambda x : ' '.join(x.split()[:2]))

In [ ]:
print(proteins_df)

In [ ]:
organisms_of_interest_df = pd.read_csv('organisms.csv',  header=0, sep=',')
#x = organisms_of_interest_df.reset_index().fillna(value=1)
display(organisms_of_interest_df)

It is important removing host organisms and vector organisms. we set the appropriate index and then we remove all vector based organisms using the drop pandas function with the help of the labels parameter.

In [ ]:
organisms_of_interest_index = organisms_of_interest_df.set_index('VEuPathDB Project')
display(organisms_of_interest_index)

In [ ]:
#removes organisms that are from VectorBase module of VEUpathDB database
organisms_of_interest_df1 = organisms_of_interest_index.drop(labels='HostDB', axis=0)
organisms_of_interest_df = organisms_of_interest_df1.drop(labels='VectorBase', axis=0).reset_index(drop=False)
display(organisms_of_interest_df)

In [ ]:
organisms_of_interest_df.count()

In [ ]:
organisms_list = list(organisms_of_interest_df['Species'])

In [ ]:
display(organisms_list)

In [ ]:
if 'Homo sapiens' in organisms_list:    #check
  print('true')
else:
  print('no')

In [ ]:
df_matches = proteins_df.loc[proteins_df['Organism'].isin(organisms_list).reset_index(drop=True)]

In [ ]:
display(df_matches)

In [ ]:
df_matches.to_csv('matches.csv', sep=',')

Text recognition module. 
after trying to create an automatic tool for correctly classify proteins taking in consideration only the code "ECO:0000269" (experimentally annotation), we decided to manually classify protein localization in the cell.

In [ ]:
secreted_list = ('Secreted' or 'Secreted.' or 'secreted')

In [ ]:
membrane_list = ('surface' or 'Cell' or 'surface.' or 'Surface.')

In [ ]:
intracell_list = ('Endoplasmic' or 'reticulum' or 'Cytoplasm' or 'Cytoplasm.' or 'Nucleus' or 'Peroxisome' or 'Golgi' or 'apparatus' or 'Mitochondrion' or
                 'inner' or 'Vacuole' or 'Cytoplasmic' or 'Peroxisome' or 'droplet' or
                 'Nucleus.' or 'nucleus' or 'nucleus.' or 'Nucleus' or 'Endosome' or 'Hydrogenosome' or 'inner' or 'Vacuole')

In [ ]:
#for s in df_matches['Subcellular location ']:       #some chunks for checking the code
  #print(s.split(' '))



In [ ]:
#replace strings to better perform the analysis
df_matches['Subcellular location'] = df_matches['Subcellular location'].replace({'SUBCELLULAR LOCATION': 'LOCATION'}, regex=True)
print(df_matches)

In [ ]:
#classification module
#we subgroup proteins due to their subcellular localization creating 4 labels (in, out, membrane, mt membrane)
#out=secreted

def selector(x):
  if membrane_list in x.split(' '):
    return 'membrane'
  
  if 'Cell' in x.split(' '):
    return 'membrane'
  if 'LOCATION: Membrane' in x:
    return 'membrane'
  if secreted_list in x.split(' '):
    return 'out'
  if 'Secreted.' in x.split(' '):
    return 'out'
  if 'membrane' and 'Vacuole' in x.split(' '):
    return 'in'
  if 'membrane' and 'vacuole' in x.split(' '):
    return 'in'
  if 'membrane' and 'Endoplasmic' and 'reticulum' in x.split(' '):
    return 'in'
  if 'Mitochondrion' and 'inner' and 'membrane' in x.split(' '):
    return 'in'
  if 'Cytoplasm' in x.split(' '):
    return 'in'
  else:
    return 'in'


In [ ]:
df_matches['Subcellular location'] = df_matches['Subcellular location'].apply(selector)

In [ ]:
display(df_matches)

In [ ]:
df_matches.to_csv('matches_final.csv', sep=',')

In [ ]:
df_matches.count()

we first created only 3 classes. after some tentatives, we decided to add a class (internal membrane). 

In [ ]:
def selector(x):
    if '0000269' in x:
        return 'validated'
    else:
        pass

df_final = df_match.copy()
df_final['Subcellular location ext'] = df_final['Subcellular location'].apply(selector)

df_final = df_final.dropna()
df_final.to_csv('matches_final.csv', sep=',')